# Базовый прогон модели SIR

**Принадлежность:** Российский университет дружбы народов

## Назначение скрипта

Данный скрипт выполняет один базовый эксперимент с фиксированными параметрами
для модели SIR, реализованной с использованием сетей Петри.

### Что делает скрипт

1. Выполняет два типа симуляции:
   - **Детерминированную** (решение ОДУ) — даёт плавную усреднённую динамику
   - **Стохастическую** (алгоритм Гиллеспи) — учитывает случайные флуктуации

2. Сохраняет результаты в CSV‑файлы

3. Строит и сохраняет графики S(t), I(t), R(t) для обоих типов симуляции

## Параметры модели

| Параметр | Значение | Описание |
|----------|----------|----------|
| β | 0.3 | Коэффициент заражения |
| γ | 0.1 | Коэффициент выздоровления |
| tmax | 100.0 | Время симуляции |
| S₀ | 990 | Начальное число восприимчивых |
| I₀ | 10 | Начальное число инфицированных |
| R₀ | 0 | Начальное число выздоровевших |

**Зерно для ГСЧ:** `Random.seed!(123)` — обеспечивает воспроизводимость стохастического прогона.

## Выходные данные

| Файл | Описание |
|------|----------|
| `data/sir_det.csv` | Детерминированная симуляция (time, S, I, R) |
| `data/sir_stoch.csv` | Стохастическая симуляция (time, S, I, R) |
| `plots/sir_det_dynamics.png` | График динамики (ОДУ) — рис. 6.1 |
| `plots/sir_stoch_dynamics.png` | График динамики (Гиллеспи) — рис. 6.2 |

## Интерпретация результатов

- **Детерминированный график** показывает классический пик эпидемии: рост I,
  максимум, затем спад до нуля; R растёт и выходит на плато, S падает.

- **Стохастический график** может иметь шумы и немного отличаться по времени пика
  и амплитуде — это демонстрирует влияние случайности.

- **Сравнение двух типов симуляции** показывает, что при большом начальном числе
  восприимчивых (990) стохастическая траектория близка к детерминированной,
  но при малых числах различия были бы значительны.

## Инициализация проекта DrWatson

In [1]:
using DrWatson
@quickactivate "project"

## Подключение генератора случайных чисел

In [2]:
using Random

## Загрузка модуля SIRPetri

Подключаем модуль с реализацией модели SIR на сетях Петри.
Файл `src/SIRPetri.jl` содержит функции:
- `build_sir_network` — создание сети
- `simulate_deterministic` — детерминированная симуляция
- `simulate_stochastic` — стохастическая симуляция
- `plot_sir` — визуализация результатов

In [3]:
include(srcdir("SIRPetri.jl"))
using .SIRPetri

## Подключение утилит для работы с данными и графикой

In [4]:
using DataFrames, CSV, Plots

## Задание параметров модели

In [5]:
β = 0.3
γ = 0.1
tmax = 100.0

100.0

## Создание сети Петри

Функция `build_sir_network` создаёт размеченную сеть с:
- Состояниями: S (Susceptible), I (Infectious), R (Recovered)
- Переходами: infection (S + I → I + I), recovery (I → R)
- Начальной маркировкой: S₀ = 990, I₀ = 10, R₀ = 0

In [6]:
net, u0, states = build_sir_network(β, γ)

(AlgebraicPetri.LabelledPetriNet:
  T = 1:2
  S = 1:3
  I = 1:3
  O = 1:3
  Name = 1:0
  it : I → T = [1, 1, 2]
  is : I → S = [1, 2, 2]
  ot : O → T = [1, 1, 2]
  os : O → S = [2, 2, 3]
  tname : T → Name = [:infection, :recovery]
  sname : S → Name = [:S, :I, :R], [990.0, 10.0, 0.0], [:S, :I, :R])

## Детерминированная симуляция

Решает систему обыкновенных дифференциальных уравнений:

```
dS/dt = -β·S·I
dI/dt =  β·S·I - γ·I
dR/dt =  γ·I
```

**Метод решения:** Tsit5 (метод Рунге-Кутты 5-го порядка)
**Шаг сохранения:** saveat = 0.5

In [7]:
df_det = simulate_deterministic(net, u0, (0.0, tmax), saveat = 0.5, rates = [β, γ])

Row,time,S,I,R
,Float64,Float64,Float64,Float64
1,0.0,990.0,10.0,0.0
2,0.5,2.47053e-7,952.691,47.3086
3,1.0,3.28252e-7,906.228,93.7719
4,1.5,-1.78365e-7,862.031,137.969
5,2.0,-2.6597e-7,819.989,180.011
6,2.5,2.389e-7,779.998,220.002
7,3.0,-3.23518e-7,741.957,258.043
8,3.5,-2.30682e-7,705.771,294.229
9,4.0,3.04028e-8,671.35,328.65


### Сохранение результатов детерминированной симуляции

In [8]:
CSV.write(datadir("sir_det.csv"), df_det)

"/home/mmulitina/work/study/2026-1/backup/2026-1-study-simulation-modeling/labs/lab06/project/data/sir_det.csv"

## Стохастическая симуляция

Использует прямой алгоритм Гиллеспи (Gillespie SSA):

1. Вычисление пропускных способностей:
   - `a_inf = β·S·I` — заражение
   - `a_rec = γ·I` — выздоровление

2. Время до следующего события: `dt = -ln(r) / (a_inf + a_rec)`

3. Случайный выбор перехода пропорционально весу

**Зерно для воспроизводимости:** 123

In [9]:
Random.seed!(123)
df_stoch = simulate_stochastic(net, u0, (0.0, tmax), rates = [β, γ])

Row,time,S,I,R
,Float64,Float64,Float64,Float64
1,0.0,990.0,10.0,0.0
2,0.000219318,989.0,11.0,0.0
3,0.00025471,988.0,12.0,0.0
4,0.000435457,987.0,13.0,0.0
5,0.00124186,986.0,14.0,0.0
6,0.00137312,985.0,15.0,0.0
7,0.00151759,984.0,16.0,0.0
8,0.00219412,983.0,17.0,0.0
9,0.00239638,982.0,18.0,0.0


### Сохранение результатов стохастической симуляции

In [10]:
CSV.write(datadir("sir_stoch.csv"), df_stoch)

"/home/mmulitina/work/study/2026-1/backup/2026-1-study-simulation-modeling/labs/lab06/project/data/sir_stoch.csv"

## Визуализация и сохранение графиков

### Рисунок 6.1: Детерминистический график динамики SIR

На графике отображаются три кривые:
- **S(t)** — восприимчивые (синяя линия)
- **I(t)** — инфицированные (красная линия)
- **R(t)** — выздоровевшие (зелёная линия)

Характерные особенности:
- Пик эпидемии (максимум I(t))
- Плато R(t) — итоговое число переболевших
- Монотонное убывание S(t)

In [11]:
p_det = plot_sir(df_det)
savefig(plotsdir("sir_det_dynamics.png"))

Could not create decoration from factory! Running with no decorations.


"/home/mmulitina/work/study/2026-1/backup/2026-1-study-simulation-modeling/labs/lab06/project/plots/sir_det_dynamics.png"

### Рисунок 6.2: Стохастический график динамики SIR

В отличие от детерминированного графика, стохастическая траектория:
- Имеет шумы и флуктуации
- Может немного отличаться по времени пика и амплитуде
- Демонстрирует влияние случайности на динамику эпидемии

In [12]:
p_stoch = plot_sir(df_stoch)
savefig(plotsdir("sir_stoch_dynamics.png"))

Could not create decoration from factory! Running with no decorations.


"/home/mmulitina/work/study/2026-1/backup/2026-1-study-simulation-modeling/labs/lab06/project/plots/sir_stoch_dynamics.png"

## Завершение работы

In [13]:
println("Базовый прогон завершён. Результаты в data/ и plots/")

Базовый прогон завершён. Результаты в data/ и plots/
